In [11]:
from pyspark.sql import functions as F
from spark_session import get_spark

spark = get_spark("FastFood-P4-Graphes",
                  extra_packages=["graphframes:graphframes:0.8.3-spark3.5-s_2.12"])
sc = spark.sparkContext

# Rend le module Python graphframes (contenu dans le jar) importable
import glob, sys
for j in glob.glob("/root/.ivy2/jars/graphframes*.jar"):
    if j not in sys.path:
        sys.path.insert(0, j)
        sc.addPyFile(j)
from graphframes import GraphFrame
from graphframes.lib import AggregateMessages as AM
print("GraphFrames OK")

GraphFrames OK


In [12]:
silver = spark.read.parquet("s3a://lake/silver/orders_history/")
delivered = silver.filter(F.col("status") == "DELIVERED")

clients = (delivered.select(F.col("customer_id").alias("id"), "city").distinct()
           .withColumn("type", F.lit("client"))
           .withColumn("category", F.lit(None).cast("string"))
           .withColumn("name", F.lit(None).cast("string")))
restos = (delivered.select(F.col("restaurant_id").alias("id"), "city", "category", "name").distinct()
          .withColumn("type", F.lit("restaurant")))

vertices = clients.unionByName(restos).cache()
edges = (delivered.groupBy(F.col("customer_id").alias("src"), F.col("restaurant_id").alias("dst"))
         .agg(F.count("*").alias("nb_commandes"),
              F.round(F.sum("amount_mad"), 2).alias("montant_total"))
         .cache())

g = GraphFrame(vertices, edges)
print("Sommets :", g.vertices.count(), "| dont restaurants :", restos.count())
print("Arêtes  :", g.edges.count())

Sommets : 7173 | dont restaurants : 235


[Stage 19:=============================>                            (2 + 2) / 4]

Arêtes  : 13047


In [13]:
print("Top 10 restaurants (clients distincts) :")
g.inDegrees.orderBy(F.desc("inDegree")).show(10)
print("Top 10 clients (restaurants distincts) :")
g.outDegrees.orderBy(F.desc("outDegree")).show(10)

Top 10 restaurants (clients distincts) :


+-----+--------+
|   id|inDegree|
+-----+--------+
|R-157|     483|
|R-188|     279|
|R-013|     231|
|R-147|     215|
|R-153|     211|
|R-012|     204|
|R-098|     195|
|R-158|     187|
|R-062|     175|
|R-011|     172|
+-----+--------+
only showing top 10 rows

Top 10 clients (restaurants distincts) :
+-------+---------+
|     id|outDegree|
+-------+---------+
|C-03378|       13|
|C-10318|       10|
|C-10879|       10|
|C-10221|       10|
|C-06677|        9|
|C-03771|        9|
|C-00870|        9|
|C-08697|        9|
|C-07025|        9|
|C-07089|        9|
+-------+---------+
only showing top 10 rows



In [14]:
print("Top 10 restaurants par nombre de clients distincts (inDegree)")
(g.inDegrees.join(restos, "id")
  .select("id", "name", "city", "category", "inDegree")
  .orderBy(F.desc("inDegree")).show(10, truncate=False))

print("Top 10 clients par nombre de restaurants différents (outDegree)")
g.outDegrees.orderBy(F.desc("outDegree")).show(10)

Top 10 restaurants par nombre de clients distincts (inDegree)


+-----+------------------------------+----------+---------+--------+
|id   |name                          |city      |category |inDegree|
+-----+------------------------------+----------+---------+--------+
|R-157|Brothers Burger - Belvédère   |Casablanca|burger   |483     |
|R-188|Smash Big - Agdal             |Rabat     |burger   |279     |
|R-013|Rôtisserie Golden - Guéliz    |Marrakech |poulet   |231     |
|R-147|Royal Tacos - Akkari          |Rabat     |tacos    |215     |
|R-153|Restaurant Chez Hamid - Maârif|Casablanca|marocain |211     |
|R-012|Pizza Napoli - Bourgogne      |Casablanca|pizza    |204     |
|R-098|Restaurant Tafilalet - Akkari |Rabat     |marocain |195     |
|R-158|Sicilia Pizza - Beni Makada   |Tanger    |pizza    |187     |
|R-062|Saigon Asian Food - Agdal     |Rabat     |asiatique|175     |
|R-011|Restaurant Lmima - Aïn Sebaâ  |Casablanca|marocain |172     |
+-----+------------------------------+----------+---------+--------+
only showing top 10 rows

Top 10 c

In [15]:
pr = g.pageRank(resetProbability=0.15, maxIter=10)
top_pr = (pr.vertices.filter("type = 'restaurant'")
            .select("id", "name", "city", "category", F.round("pagerank", 3).alias("pagerank"))
            .orderBy(F.desc("pagerank")))
top_pr.show(10, truncate=False)

nb_cmd = edges.groupBy(F.col("dst").alias("id")).agg(F.sum("nb_commandes").alias("nb_livrees"))
(top_pr.limit(10).join(nb_cmd, "id").join(g.inDegrees, "id")
       .orderBy(F.desc("pagerank")).show(truncate=False))

+-----+------------------------------+----------+---------+--------+
|id   |name                          |city      |category |pagerank|
+-----+------------------------------+----------+---------+--------+
|R-157|Brothers Burger - Belvédère   |Casablanca|burger   |125.286 |
|R-188|Smash Big - Agdal             |Rabat     |burger   |72.601  |
|R-013|Rôtisserie Golden - Guéliz    |Marrakech |poulet   |61.832  |
|R-153|Restaurant Chez Hamid - Maârif|Casablanca|marocain |56.833  |
|R-147|Royal Tacos - Akkari          |Rabat     |tacos    |54.281  |
|R-158|Sicilia Pizza - Beni Makada   |Tanger    |pizza    |51.562  |
|R-012|Pizza Napoli - Bourgogne      |Casablanca|pizza    |50.67   |
|R-098|Restaurant Tafilalet - Akkari |Rabat     |marocain |48.058  |
|R-062|Saigon Asian Food - Agdal     |Rabat     |asiatique|45.897  |
|R-011|Restaurant Lmima - Aïn Sebaâ  |Casablanca|marocain |44.18   |
+-----+------------------------------+----------+---------+--------+
only showing top 10 rows

+-----+-

In [16]:
sc.setCheckpointDir("s3a://lake/checkpoints/graphframes")
cc = g.connectedComponents()
tailles = cc.groupBy("component").agg(F.count("*").alias("taille"),
                                       F.collect_set("city").alias("villes"))
print("Nombre de composantes :", tailles.count())
tailles.orderBy(F.desc("taille")).show(truncate=False)

Nombre de composantes : 12


[Stage 868:==================================>                    (10 + 2) / 16]

+-----------+------+------------+
|component  |taille|villes      |
+-----------+------+------------+
|0          |3178  |[Casablanca]|
|6          |1614  |[Rabat]     |
|1          |1315  |[Marrakech] |
|3          |1049  |[Tanger]    |
|172        |3     |[Marrakech] |
|34359739078|2     |[Rabat]     |
|892        |2     |[Rabat]     |
|8589935452 |2     |[Marrakech] |
|473        |2     |[Marrakech] |
|25769804398|2     |[Rabat]     |
|733        |2     |[Marrakech] |
|8589935446 |2     |[Casablanca]|
+-----------+------+------------+



In [17]:
motifs = (g.find("(c)-[e1]->(r1); (c)-[e2]->(r2)")
           .filter("r1.category = r2.category AND r1.id < r2.id"))
print("Couples (client, 2 restaurants de même catégorie) :", motifs.count())
motifs.select("c.id", "r1.id", "r1.name", "r2.id", "r2.name", "r1.category").show(10, truncate=False)

pairs = motifs.select(F.col("r1.id").alias("r1"), F.col("r2.id").alias("r2"))
co = (pairs.unionByName(pairs.select(F.col("r2").alias("r1"), F.col("r1").alias("r2")))
           .groupBy("r1", "r2").agg(F.count("*").alias("score")))

client = motifs.select("c.id").first()[0]
deja = edges.filter(F.col("src") == client).select(F.col("dst").alias("r1"))
reco = (co.join(deja, "r1")
          .groupBy("r2").agg(F.sum("score").alias("score"))
          .join(deja.withColumnRenamed("r1", "r2"), "r2", "left_anti")
          .join(restos.select(F.col("id").alias("r2"), "name", "category"), "r2")
          .orderBy(F.desc("score")))
print(f"Client {client} a déjà commandé chez :", [r[0] for r in deja.collect()])
print("Recommandations :")
reco.show(5, truncate=False)

Couples (client, 2 restaurants de même catégorie) : 1112
+-------+-----+----------------------------------+-----+-------------------------------+--------+
|id     |id   |name                              |id   |name                           |category|
+-------+-----+----------------------------------+-----+-------------------------------+--------+
|C-03957|R-033|Tajine Bab Doukkala - Sidi Maârouf|R-153|Restaurant Chez Hamid - Maârif |marocain|
|C-06965|R-011|Restaurant Lmima - Aïn Sebaâ      |R-127|Dar Zitouna - Aïn Sebaâ        |marocain|
|C-03610|R-010|Smash Grill - Aïn Sebaâ           |R-083|Dakhla Burger - Derb Sultan    |burger  |
|C-10759|R-047|Tacos Lyon - Sidi Maârouf         |R-055|Avenue Tacos - Aïn Diab        |tacos   |
|C-02840|R-008|Smash Street - Bourgogne          |R-010|Smash Grill - Aïn Sebaâ        |burger  |
|C-02840|R-008|Smash Street - Bourgogne          |R-157|Brothers Burger - Belvédère    |burger  |
|C-05096|R-153|Restaurant Chez Hamid - Maârif    |R-183|Cousc

[Stage 923:================================================>      (14 + 2) / 16]

+-----+-----+----------------------------------+--------+
|r2   |score|name                              |category|
+-----+-----+----------------------------------+--------+
|R-011|16   |Restaurant Lmima - Aïn Sebaâ      |marocain|
|R-046|14   |Restaurant Chez Hamid - Gauthier  |marocain|
|R-115|11   |Riad Lmima - Hay Mohammadi        |marocain|
|R-089|7    |Restaurant Al Andalous - Aïn Sebaâ|marocain|
|R-183|6    |Couscous Essaouira - Bourgogne    |marocain|
+-----+-----+----------------------------------+--------+
only showing top 5 rows



In [18]:
ca_msg = g.aggregateMessages(F.sum(AM.msg).alias("ca_recu"),
                             sendToDst=AM.edge["montant_total"])
ca_msg = ca_msg.join(restos.select("id", "name"), "id")
ca_msg.orderBy(F.desc("ca_recu")).show(10, truncate=False)

ca_groupby = edges.groupBy(F.col("dst").alias("id")).agg(F.sum("montant_total").alias("ca_groupby"))
diff = (ca_msg.join(ca_groupby, "id")
              .filter(F.abs(F.col("ca_recu") - F.col("ca_groupby")) > 0.01).count())
print("Restaurants avec un écart aggregateMessages / groupBy :", diff)

+-----+-------+------------------------------+
|id   |ca_recu|name                          |
+-----+-------+------------------------------+
|R-157|85057.0|Brothers Burger - Belvédère   |
|R-188|55491.0|Smash Big - Agdal             |
|R-062|39869.0|Saigon Asian Food - Agdal     |
|R-012|39838.5|Pizza Napoli - Bourgogne      |
|R-158|39012.0|Sicilia Pizza - Beni Makada   |
|R-013|38911.5|Rôtisserie Golden - Guéliz    |
|R-109|38332.0|Maki Osaka - Hay Mohammadi    |
|R-153|35875.0|Restaurant Chez Hamid - Maârif|
|R-130|34529.5|Maki Sakura - Centre-ville    |
|R-098|34254.0|Restaurant Tafilalet - Akkari |
+-----+-------+------------------------------+
only showing top 10 rows

Restaurants avec un écart aggregateMessages / groupBy : 0


In [19]:
spark.stop()

## Partie 4 – Analyse

### Le graphe
7 173 sommets (6 938 clients, 235 restaurants) et 13 047 arêtes orientées client → restaurant, pondérées par le nombre de commandes et le montant total. Parmi les 235 restaurants, **35 sont absents du référentiel** (identifiants R-9xx) : les commandes qui les visent sont conservées (jointure gauche), mais ces sommets n'ont ni nom ni catégorie. Le graphe est **biparti** : aucune arête client–client ni restaurant–restaurant.

### Degrés
- **inDegree** (clients distincts par restaurant) : R-157 Brothers Burger (483) domine largement, devant R-188 Smash Big (279) et R-013 Rôtisserie Golden (231).
- **outDegree** (restaurants différents par client) : 13 au maximum (C-03378), puis plusieurs clients à 10 et 9. La plupart des clients restent fidèles à quelques adresses de leur ville.

### PageRank et comparaison avec le top 5 de la Partie 2
| Rang | Partie 2 (commandes livrées) | PageRank |
|---|---|---|
| 1 | R-157 (572) | R-157 (125,3) |
| 2 | R-188 (357) | R-188 (72,6) |
| 3 | R-013 (282) | R-013 (61,8) |
| 4 | R-147 (245) | **R-153 (56,8)** |
| 5 | R-012 (243) | R-147 (54,3) |

Les deux classements ne mesurent pas la même chose :
- La Partie 2 compte le **volume** de commandes : un client qui commande 10 fois pèse 10 fois.
- PageRank ignore le poids des arêtes : un client compte **une seule fois** par restaurant, et il transmet son score **divisé par son nombre de restaurants** (outDegree). Un restaurant est donc bien classé s'il a beaucoup de clients **distincts** et **fidèles** (qui fréquentent peu d'autres restaurants).
- Exemple : R-153 n'a que 240 livraisons et 211 clients distincts, contre 245 livraisons et 215 clients pour R-147, mais il le dépasse au PageRank (56,8 contre 54,3) : ses clients sont plus exclusifs, leur vote est moins dilué. De même, R-012 (243 livraisons) sort du top 5 et passe derrière R-158 (230 livraisons) : la clientèle de R-158, à Tanger, a moins d'alternatives.

PageRank mesure donc l'**attachement de la clientèle** plutôt que le volume de ventes.

### Composantes connexes
12 composantes :
- **4 grandes, une par ville** : Casablanca (3 178 sommets), Rabat (1 614), Marrakech (1 315), Tanger (1 049). Un client ne commande que dans sa ville, donc aucune arête ne relie deux villes : ce sont 4 marchés indépendants, et la réaffectation des livreurs se raisonne ville par ville.
- **8 petites composantes de 2 ou 3 sommets** : ce sont des clients qui n'ont commandé que dans un **restaurant hors référentiel** (R-907, R-941, R-962…). L'analyse de graphe fait ainsi ressortir un problème de qualité des données : des commandes vers des restaurants inconnus du référentiel.

`connectedComponents()` exige un répertoire de checkpoint (`sc.setCheckpointDir`) : l'algorithme est itératif, et le checkpoint tronque le lineage, qui grandirait à chaque itération.

### Motifs et recommandation
Le motif `(c)-[e1]->(r1); (c)-[e2]->(r2)` filtré sur `r1.category = r2.category` donne **1 112 couples** de restaurants concurrents partageant un client. On en déduit une **co-fréquentation** : « les clients de r1 vont aussi chez r2 ». Le client C-03957 a déjà commandé chez R-033 et R-153 (cuisine marocaine). On lui recommande les restaurants marocains les plus co-fréquentés avec ceux qu'il connaît et qu'il n'a pas encore essayés : **R-011 Restaurant Lmima** (score 16), R-046 (14) et R-115 (11).

### aggregateMessages
Chaque arête envoie son `montant_total` vers sa destination (`sendToDst`), et chaque restaurant somme les messages reçus (`F.sum(AM.msg)`). R-157 reçoit 85 057 MAD, devant R-188 (55 491) et R-062 (39 869). Le résultat est identique au `groupBy("dst").sum()`, avec **0 écart**. On remarque que le classement par chiffre d'affaires diffère encore des deux autres : R-062 (asiatique) est 3e en CA mais seulement 9e au PageRank, car son panier moyen est élevé.

### GraphX ou GraphFrames
| | GraphX | GraphFrames |
|---|---|---|
| API | RDD, `Graph[VD, ED]` typé | DataFrames `vertices(id, …)` / `edges(src, dst, …)` |
| Langage | Scala/Java uniquement | Scala, Java, **Python** |
| Optimisation | aucune (lambdas opaques) | **Catalyst** et Tungsten sur les requêtes |
| Requêtes | algorithmes et `Pregel` | algorithmes, **recherche de motifs** (`find`), filtres SQL |

**GraphFrames est utilisé en PySpark** car GraphX n'a pas d'API Python. GraphFrames encapsule GraphX : PageRank et les composantes connexes appellent GraphX sous le capot, mais l'utilisateur manipule des DataFrames.

### Le modèle Pregel
Pregel est un modèle de calcul **« penser comme un sommet »**, organisé en **super-étapes** synchronisées :
1. chaque sommet actif **envoie des messages** le long de ses arêtes ;
2. chaque sommet **agrège** les messages reçus (somme, min…) ;
3. chaque sommet **met à jour son état** à partir de cet agrégat ;
4. on recommence jusqu'à convergence ou jusqu'à `maxIter`.

- **PageRank** : à chaque itération, un sommet envoie `rang / outDegree` à ses voisins, et chacun calcule `rang = 0.15 + 0.85 × Σ messages`. Ici, 10 itérations.
- **aggregateMessages** est **une seule super-étape** Pregel : envoi le long des arêtes, puis agrégation à la destination.
- **connectedComponents** propage le plus petit identifiant de composante jusqu'à stabilité.